# 01 — Nettoyage

## Partie 1 — Charger et comparer les formats (ticket SM-004)

> **De : Mame Diarra.** « On me dit CSV, JSON, Parquet. Je veux savoir lequel prendre pour quoi, avec des chiffres. »

**L'idée** : un format se juge à l'usage, selon quatre critères :

| Critère | La question |
|---|---|
| Temps de lecture | combien de temps pour charger le fichier dans Pandas ? |
| Poids sur disque | combien de place prend-il ? |
| Fidélité des types | relit-on exactement les types écrits (entiers, dates) ? |
| Lisibilité | peut-on l'ouvrir et le lire à l'œil ? |

Sur 10 000 lignes, les différences sont minuscules : on mesure donc sur la **partition volume** (200 000 lignes de commande).

**Avant de lancer ce notebook**, la partition volume doit exister :
```bash
python -m senmarket.generateur --run-date 2026-11-02 --rows 200000 --defauts tous
```

**Plan**
1. Charger la table `lignes_commande` brute
2. L'exporter en 4 formats
3. Mesurer : temps, taille, types
4. Le type des dates selon le format (`commandes`)
5. Lire par morceaux (`chunksize`) et vérifier qu'on obtient la même chose
6. Conclure : quel format pour quel usage ?

In [2]:
import time
from pathlib import Path

import numpy as np
import pandas as pd

from senmarket.chemins import dossier_donnees, dossier_raw

RUN_DATE = "2026-11-02"          # la partition volume (200 000 lignes de commande)
SOURCE = dossier_raw(RUN_DATE)

if not SOURCE.exists():
    raise FileNotFoundError(
        f"La partition {SOURCE} n'existe pas. Lance d'abord dans le terminal :\n"
        f"python -m senmarket.generateur --run-date {RUN_DATE} --rows 200000 --defauts tous"
    )

# Les exports de comparaison vont dans un dossier à part, JAMAIS à côté des sources
SORTIE = dossier_donnees() / "comparaison_formats" / f"date={RUN_DATE}"
SORTIE.mkdir(parents=True, exist_ok=True)
print("Source :", SOURCE)
print("Sortie :", SORTIE)

Source : /home/mame-diarra/senmarket/data/raw/date=2026-11-02
Sortie : /home/mame-diarra/senmarket/data/comparaison_formats/date=2026-11-02


## 1. Charger la table `lignes_commande` brute

C'est la plus grosse table « métier » (la navigation est plus grosse, mais moins utile à l'analyse).

In [3]:
lignes = pd.read_csv(SOURCE / "lignes_commande.csv")
print(lignes.shape)
print(lignes.dtypes)
lignes.head()

(210000, 6)
ligne_commande_id      str
commande_id            str
produit_id             str
quantite             int64
prix_unitaire_xof    int64
montant_xof            str
dtype: object


,ligne_commande_id,commande_id,produit_id,quantite,prix_unitaire_xof,montant_xof
0,LIG0000001,CMD0000001,PRD0000030,2,18500,37 000 FCFA
1,LIG0000002,CMD0000001,PRD0000007,2,13699,27398
2,LIG0000003,CMD0000001,PRD0000021,4,12500,NaN
3,LIG0000003,CMD0000001,PRD0000021,4,12500,NaN
4,LIG0000004,CMD0000001,PRD0000040,2,37379,74758


👀 **Observe** : `montant_xof` est de type `str`, et non entier. C'est la conséquence des montants écrits en texte (« 45 000 FCFA ») injectés au chapitre 2 : il suffit d'**une** valeur texte pour que Pandas lise toute la colonne comme du texte. On la nettoiera dans la partie 2 ; ici, on compare les formats sur les données telles qu'elles arrivent.

## 2. Exporter en 4 formats

| Format | Ce que c'est |
|---|---|
| **CSV** | texte, une ligne par enregistrement, valeurs séparées par des virgules |
| **JSON (tableau)** | un seul grand tableau `[{...}, {...}]` : il faut tout lire pour avoir la première ligne |
| **JSON Lines** | un objet JSON par ligne : on peut lire et traiter ligne par ligne (« en flux ») |
| **Parquet** | binaire, colonnaire, compressé, et **typé** |

In [4]:
fichiers = {
    "CSV":        SORTIE / "lignes_commande.csv",
    "JSON":       SORTIE / "lignes_commande.json",
    "JSON Lines": SORTIE / "lignes_commande.jsonl",
    "Parquet":    SORTIE / "lignes_commande.parquet",
}

lignes.to_csv(fichiers["CSV"], index=False)
lignes.to_json(fichiers["JSON"], orient="records", force_ascii=False)
lignes.to_json(fichiers["JSON Lines"], orient="records", lines=True, force_ascii=False)
lignes.to_parquet(fichiers["Parquet"], index=False)

for nom, chemin in fichiers.items():
    print(f"{nom:<11} {chemin.name}")

CSV         lignes_commande.csv
JSON        lignes_commande.json
JSON Lines  lignes_commande.jsonl
Parquet     lignes_commande.parquet


In [5]:
# À quoi ressemble le début de chaque fichier texte ?
print("CSV :")
print(fichiers["CSV"].read_text(encoding="utf-8")[:160])
print("\nJSON (tableau) :")
print(fichiers["JSON"].read_text(encoding="utf-8")[:160])
print("\nJSON Lines :")
print(fichiers["JSON Lines"].read_text(encoding="utf-8")[:160])

CSV :
ligne_commande_id,commande_id,produit_id,quantite,prix_unitaire_xof,montant_xof
LIG0000001,CMD0000001,PRD0000030,2,18500,37 000 FCFA
LIG0000002,CMD0000001,PRD00

JSON (tableau) :
[{"ligne_commande_id":"LIG0000001","commande_id":"CMD0000001","produit_id":"PRD0000030","quantite":2,"prix_unitaire_xof":18500,"montant_xof":"37 000 FCFA"},{"li

JSON Lines :
{"ligne_commande_id":"LIG0000001","commande_id":"CMD0000001","produit_id":"PRD0000030","quantite":2,"prix_unitaire_xof":18500,"montant_xof":"37 000 FCFA"}
{"lig


💡 Remarque que JSON répète le nom de chaque colonne **sur chaque ligne** : c'est pour cela qu'il est verbeux. Le Parquet, lui, est binaire : l'ouvrir dans un éditeur de texte n'affiche rien de lisible.

## 3. Mesurer : temps, taille, types

⚠️ **Le piège du cache.** La première fois qu'on lit un fichier, le système le charge depuis le disque (lecture « à froid »). Ensuite, il le garde en mémoire vive (RAM) : les lectures suivantes sont beaucoup plus rapides. Comparer une lecture à froid avec une lecture en cache serait injuste.

**La règle du ticket** : pour chaque format, une première lecture **ignorée** (elle met tout le monde en cache), puis **5 mesures**, dont on garde la **médiane** (la valeur du milieu, insensible à une mesure exceptionnellement lente).

In [6]:
lecteurs = {
    "CSV":        lambda p: pd.read_csv(p),
    "JSON":       lambda p: pd.read_json(p, orient="records"),
    "JSON Lines": lambda p: pd.read_json(p, orient="records", lines=True),
    "Parquet":    lambda p: pd.read_parquet(p),
}


def mesurer(lire, chemin, repetitions=5):
    """Temps médian de lecture (en secondes), après une lecture d'échauffement ignorée."""
    lire(chemin)                                  # lecture à froid : ignorée
    temps = []
    for _ in range(repetitions):
        debut = time.perf_counter()
        lire(chemin)
        temps.append(time.perf_counter() - debut)
    return float(np.median(temps))


resultats = []
for nom, chemin in fichiers.items():
    df = lecteurs[nom](chemin)
    resultats.append({
        "format": nom,
        "temps_median_s": round(mesurer(lecteurs[nom], chemin), 3),
        "taille_mo": round(chemin.stat().st_size / 1024**2, 2),
        "type_quantite": str(df["quantite"].dtype),
        "type_montant_xof": str(df["montant_xof"].dtype),
    })
    print(f"{nom} mesuré")

comparaison = pd.DataFrame(resultats).set_index("format")
comparaison["x_plus_lent_que_parquet"] = (comparaison["temps_median_s"]
                                          / comparaison.loc["Parquet", "temps_median_s"]).round(1)
comparaison["x_plus_lourd_que_parquet"] = (comparaison["taille_mo"]
                                           / comparaison.loc["Parquet", "taille_mo"]).round(1)
comparaison

CSV mesuré
JSON mesuré
JSON Lines mesuré
Parquet mesuré


,temps_median_s,taille_mo,type_quantite,type_montant_xof,x_plus_lent_que_parquet,x_plus_lourd_que_parquet
format,,,,,,
CSV,0.176,9.31,int64,str,8.4,3.0
JSON,0.441,29.77,int64,str,21.0,9.7
JSON Lines,0.523,29.77,int64,str,24.9,9.7
Parquet,0.021,3.08,int64,str,1.0,1.0


✍️ **Interprète avec tes chiffres** : quel format est le plus rapide ? Le plus léger ? De combien ? Écris-le en une phrase dans la cellule ci-dessous.

*Le parquet est le plus rapide.*

## 4. Le type des dates selon le format

La question (1) du ticket porte sur `date_commande`, qui est dans la table `commandes`. On convertit d'abord la colonne en vraie date, on l'écrit dans les 4 formats, puis on regarde le type obtenu **à la relecture**.

In [7]:
commandes = pd.read_csv(SOURCE / "commandes.csv", parse_dates=["date_commande"])
print("Type avant écriture :", commandes["date_commande"].dtype)

f_cmd = {
    "CSV":        SORTIE / "commandes.csv",
    "JSON":       SORTIE / "commandes.json",
    "JSON Lines": SORTIE / "commandes.jsonl",
    "Parquet":    SORTIE / "commandes.parquet",
}
commandes.to_csv(f_cmd["CSV"], index=False)
commandes.to_json(f_cmd["JSON"], orient="records", force_ascii=False, date_format="iso")
commandes.to_json(f_cmd["JSON Lines"], orient="records", lines=True, force_ascii=False, date_format="iso")
commandes.to_parquet(f_cmd["Parquet"], index=False)

types_dates = pd.DataFrame({
    "type_relu": {nom: str(lecteurs[nom](chemin)["date_commande"].dtype) for nom, chemin in f_cmd.items()}
})
types_dates

Type avant écriture : datetime64[us]


,type_relu
CSV,str
JSON,str
JSON Lines,str
Parquet,datetime64[us]


👀 **Ce qu'on voit** : seul le **Parquet** rend une vraie date (`datetime64`). Le CSV et le JSON rendent du texte (`str`).

**Pourquoi** : le CSV et le JSON sont du **texte** ; ils ne stockent que des caractères comme « 2026-03-14 », sans dire que c'est une date. Pandas doit le deviner, et par prudence il ne le fait pas (sauf si on le lui demande avec `parse_dates`). Le Parquet, lui, enregistre le **type** de chaque colonne avec les données : on relit exactement ce qu'on a écrit.

C'est la réponse à la question (1) du ticket.

## 5. Lire par morceaux (`chunksize`)

Quand un fichier est trop gros pour la mémoire, on le lit par morceaux. Le ticket demande de **montrer que le résultat est identique** à une lecture complète.

In [8]:
morceaux = []
for i, morceau in enumerate(pd.read_csv(fichiers["CSV"], chunksize=50_000)):
    morceaux.append(morceau)
    print(f"morceau {i} : {len(morceau)} lignes")

recolle = pd.concat(morceaux)
complet = pd.read_csv(fichiers["CSV"])

pd.testing.assert_frame_equal(recolle, complet)    # lève une erreur à la moindre différence
print("\n✅ Lecture par morceaux identique à la lecture complète :", recolle.shape)

morceau 0 : 50000 lignes
morceau 1 : 50000 lignes
morceau 2 : 50000 lignes
morceau 3 : 50000 lignes
morceau 4 : 10000 lignes

✅ Lecture par morceaux identique à la lecture complète : (210000, 6)


💡 Une différence subtile : chaque morceau devine ses types **séparément**. Si un morceau ne contenait que des montants numériques et un autre des montants en texte, leurs types `montant_xof` différeraient, et le recollage donnerait une colonne mélangée. C'est une raison de plus pour fixer les types à la lecture (`dtype=...`) sur les gros fichiers.

## 6. Conclure : quel format pour quel usage ?

Le ticket demande une conclusion par usage, **appuyée sur tes chiffres** (« Parquet est rapide » n'est pas un argument ; « sur ma machine, Parquet se lit 6 fois plus vite que le CSV » en est un).

| Usage | Format conseillé | Argument (à compléter avec tes chiffres) |
|---|---|---|
| **Lire à l'œil** (vérifier un fichier, l'ouvrir dans Excel) | CSV | lisible partout, sans outil ; … |
| **Ingérer en flux** (recevoir des données au fil de l'eau, d'une API) | JSON Lines | une ligne = un enregistrement complet : on traite au fur et à mesure, sans attendre la fin du fichier ; … |
| **Analyser** (relire souvent, filtrer, agréger) | Parquet | … fois plus rapide que le CSV, types conservés |
| **Archiver** (stocker longtemps) | Parquet | … fois plus léger que le CSV, types conservés |

✍️ Complète les « … » avec les valeurs de ton tableau de la partie 3.

## Pour `docs/MESURES.md`

La cellule suivante produit le tableau au format Markdown, prêt à copier dans `docs/MESURES.md` (section « Formats »), avec la description de la machine.

In [10]:
lignes_md = ["| Format | Temps médian (5 mesures) | Taille | Type de date_commande |", "|---|---|---|---|"]
for nom, ligne in comparaison.iterrows():
    lignes_md.append(f"| {nom} | {ligne['temps_median_s']} s | {ligne['taille_mo']} Mo | "
                     f"{types_dates.loc[nom, 'type_relu']} |")
print(f"Mesures sur lignes_commande, partition {RUN_DATE} ({len(lignes)} lignes), "
      "1 lecture d'échauffement ignorée puis médiane de 5 mesures.\n")
print("\n".join(lignes_md))

Mesures sur lignes_commande, partition 2026-11-02 (210000 lignes), 1 lecture d'échauffement ignorée puis médiane de 5 mesures.

| Format | Temps médian (5 mesures) | Taille | Type de date_commande |
|---|---|---|---|
| CSV | 0.176 s | 9.31 Mo | str |
| JSON | 0.441 s | 29.77 Mo | str |
| JSON Lines | 0.523 s | 29.77 Mo | str |
| Parquet | 0.021 s | 3.08 Mo | datetime64[us] |


  # Partie 2 — Nettoyer (ticket SM-005)

  ## Étape 4.1 — clean_montant

In [12]:
def clean_montant(serie: pd.Series) -> pd.Series:
    """Convertit les montants bruts (« 45 000 FCFA », « 45.000 XOF »…) en entiers."""
    texte = serie.astype("str")
    texte = texte.str.replace("F CFA", "", regex=False)
    texte = texte.str.replace("FCFA", "", regex=False)
    texte = texte.str.replace("XOF", "", regex=False)
    texte = texte.str.replace(" ", "", regex=False)
   
    return pd.to_numeric(texte, errors="coerce").astype("Int64")


essai = pd.Series(["37000", "45 000 FCFA", "45000 F CFA", "45.000 XOF", "-64396", None, "bonjour"])
print(clean_montant(essai))

0     37000
1     45000
2     45000
3        45
4    -64396
5      <NA>
6      <NA>
dtype: Int64
